# 00 - Template Connectivity Check

Proves this Jupyter kernel reaches both the Spark cluster and postgres,
via JDBC (from PySpark) and directly (via `psycopg2`), before any
assessment notebook relies on either path.
See `docs/features/07-jupyter-notebook-workspace-setup.md` -> Design ->
template notebook design.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]
SOURCE_TABLE = "src_transaction_daily"


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("00-template-connectivity-check")
    .getOrCreate()
)

spark_check_count = spark.range(1000).count()
spark_status = "PASS" if spark_check_count == 1000 else "FAIL"
print(f"[{spark_status}] spark connectivity: spark.range(1000).count()={spark_check_count}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/30 08:24:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[PASS] spark connectivity: spark.range(1000).count()=1000


In [3]:
jdbc_df = spark.read.jdbc(
    url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
    table=SOURCE_TABLE,
    properties={
        "user": POSTGRES_USER,
        "password": POSTGRES_PASSWORD,
        "driver": "org.postgresql.Driver",
    },
)
jdbc_row_count = jdbc_df.count()
jdbc_status = "PASS" if jdbc_row_count > 0 else "FAIL"
print(f"[{jdbc_status}] postgres JDBC path: {SOURCE_TABLE} row_count={jdbc_row_count}")


[PASS] postgres JDBC path: src_transaction_daily row_count=2010


In [4]:
conn = psycopg2.connect(
    host="postgres", port=5432, dbname=POSTGRES_DB, user=POSTGRES_USER, password=POSTGRES_PASSWORD,
)
with conn, conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {SOURCE_TABLE};")
    psycopg2_row_count = cur.fetchone()[0]
conn.close()

psycopg2_status = "PASS" if psycopg2_row_count > 0 else "FAIL"
print(f"[{psycopg2_status}] postgres psycopg2 path: {SOURCE_TABLE} row_count={psycopg2_row_count}")


[PASS] postgres psycopg2 path: src_transaction_daily row_count=2010


In [5]:
checks = {
    "spark_connectivity": spark_status,
    "postgres_jdbc": jdbc_status,
    "postgres_psycopg2": psycopg2_status,
    "row_count_cross_check": "PASS" if jdbc_row_count == psycopg2_row_count else "FAIL",
}

for name, status in checks.items():
    print(f"[{status}] {name}")

overall = "PASS" if all(s == "PASS" for s in checks.values()) else "FAIL"
print(f"[{overall}] 00-template-connectivity-check: overall status={overall}")

spark.stop()


[PASS] spark_connectivity
[PASS] postgres_jdbc
[PASS] postgres_psycopg2
[PASS] row_count_cross_check
[PASS] 00-template-connectivity-check: overall status=PASS
